# 05 · Evaluation

Phase 6: score the regex baseline against hand labels on 20 chunks from the cleaned slice.

- **Sample:** `data/eval_sample.csv`, chosen by `select_eval_sample` in `src/evaluate.py`
- **Labelling rules:** `docs/labelling_guide.md`, written before any labelling
- **Labels:** `data/labels.csv`, one row per entity, copied exactly from the chunk text
- **Who labelled:** labels were drafted by an AI assistant (Claude) from the guide by reading the chunk text, without looking at the baseline's output, and then reviewed by the author against the guide.

In [1]:
import sys
sys.path.append("../src")

import pandas as pd
from data import load_case_law, load_slice, clean_slice
from evaluate import select_eval_sample, gold_spans, predicted_spans, match, score

In [2]:
case_law = load_case_law()
df, _ = clean_slice(load_slice(case_law), case_law)

sample = pd.read_csv("../data/eval_sample.csv").merge(df[["chunk_id", "text"]], on="chunk_id")
labels = pd.read_csv("../data/labels.csv", keep_default_na=False)

## 1. The evaluation sample

20 chunks, drawn with seed 42:
- **L01–L12:** 12 chunks where the baseline found at least one entity. These test whether its matches are right (precision).
- **L13–L20:** 8 chunks where it found nothing. These show what it misses (recall).

A simple random draw of 20 would give only about 8 chunks with any match (153 of 381 chunks have one), which is too few to measure precision. The cost of this split is a bias, discussed in section 5.

This re-runs the selection to confirm the saved sample is the one the code produces:

In [3]:
reselected = select_eval_sample(df)
(reselected["chunk_id"] == sample["chunk_id"]).all()

np.True_

## 2. The labels

In [4]:
labels.groupby("type").size()

type
CASE_REF    19
DATE        11
MONEY        7
dtype: int64

In [5]:
labels.groupby("item").size().reindex(sample["item"], fill_value=0).to_frame("entities").T

item,L01,L02,L03,L04,L05,L06,L07,L08,L09,L10,L11,L12,L13,L14,L15,L16,L17,L18,L19,L20
entities,1,7,3,1,1,1,5,1,6,1,2,4,1,0,0,0,1,0,1,1


37 entities in total. Four chunks (L14, L15, L16, L18) contain none, which is normal for passages of legal argument.

Each label is then located by character position, so it can be compared with the baseline's spans:

In [6]:
gold = gold_spans(sample, labels)
pred = predicted_spans(sample)

print("gold spans:     ", len(gold), "(should equal the number of label rows:", len(labels), ")")
print("predicted spans:", len(pred))

gold spans:      37 (should equal the number of label rows: 37 )
predicted spans: 25


## 3. Scores

Two ways of counting a prediction as correct:
- **Exact:** it has the same start and end as a gold span of the same type. This is strict: `Case E018 of 2025` does **not** match the gold `Land Case E018 of 2025`.
- **Partial:** it overlaps a gold span of the same type. This is lenient: the example above counts as correct.

Each gold span can be matched by at most one prediction. Precision = TP / predicted, recall = TP / gold, and F1 is their harmonic mean.

In [7]:
exact = score(gold, pred, "exact")
exact

,gold,predicted,TP,FP,FN,precision,recall,F1
type,,,,,,,,
DATE,11,7,7,0,4,1.00,0.64,0.78
MONEY,7,6,5,1,2,0.83,0.71,0.77
CASE_REF,19,12,7,5,12,0.58,0.37,0.45
ALL,37,25,19,6,18,0.76,0.51,0.61


In [8]:
partial = score(gold, pred, "partial")
partial

,gold,predicted,TP,FP,FN,precision,recall,F1
type,,,,,,,,
DATE,11,7,7,0,4,1.0,0.64,0.78
MONEY,7,6,6,0,1,1.0,0.86,0.92
CASE_REF,19,12,12,0,7,1.0,0.63,0.77
ALL,37,25,25,0,12,1.0,0.68,0.81


In [9]:
pd.DataFrame({
    "exact F1": exact["F1"],
    "partial F1": partial["F1"],
    "gap": (partial["F1"] - exact["F1"]).round(2),
})

,exact F1,partial F1,gap
type,,,
DATE,0.78,0.78,0.00
MONEY,0.77,0.92,0.15
CASE_REF,0.45,0.77,0.32
ALL,0.61,0.81,0.20


What the numbers say:
- **Precision is high and recall is low.** When the baseline fires it's usually on a real entity, but it misses about half of them (exact recall 0.51).
- **Dates:** every predicted date is correct, and the exact and partial scores are the same. All 4 misses are one format (next section).
- **Money:** the partial score looks good (0.92), but partial matching hides a real error. `Kshs 4, 000` is extracted as `Kshs 4`, which overlaps the gold span and so counts as correct, yet the amount is wrong by a factor of 1,000.
- **Case references are the weakest type** (exact F1 0.45). 5 of the 12 predictions are incomplete spans, and 7 citations aren't found at all.

## 4. Every error, listed

In [10]:
gold_e, pred_e = match(gold, pred, "exact")
gold_p, pred_p = match(gold, pred, "partial")

print("MISSED (no overlapping prediction at all):")
gold_p.loc[~gold_p["matched"], ["item", "type", "text"]]

MISSED (no overlapping prediction at all):


,item,type,text
5,L02,DATE,24 th February 2025
8,L03,CASE_REF,(2007] KECA 115 [KLR)
10,L03,CASE_REF,[2024) KEELC 1505 (KLR])
17,L07,DATE,14 th February 2020
21,L09,MONEY,9.000
28,L11,DATE,"20 th July, 2015"
29,L12,CASE_REF,[1931] 47 TLK 557
31,L12,CASE_REF,[2003] 2 EA 519
33,L13,CASE_REF,[1972] ALL ER 606
34,L17,CASE_REF,[2004] 2 EA 163


In [11]:
print("INCOMPLETE (overlaps a gold span but the boundaries differ):")
incomplete = pred_e[~pred_e["matched"] & pred_p["matched"]].copy()
incomplete["gold"] = [
    gold[(gold["item"] == r.item) & (gold["type"] == r.type) & (gold["start"] < r.end) & (gold["end"] > r.start)]["text"].iloc[0]
    for r in incomplete.itertuples()
]
incomplete[["item", "type", "text", "gold"]].rename(columns={"text": "predicted"})

INCOMPLETE (overlaps a gold span but the boundaries differ):


,item,type,predicted,gold
1,L02,CASE_REF,Case E018 of 2025,Land Case E018 of 2025
5,L02,CASE_REF,Case E018 of 2025,Land Case E018 of 2025
7,L03,CASE_REF,[1992] KECA 42,[1992] KECA 42 (KLR]
11,L07,CASE_REF,Civil Case No. 3590 of 1995,High Court Civil Case No. 3590 of 1995
13,L07,CASE_REF,Civil Case No. 3590 of 1995,High Court Civil Case No. 3590 of 1995
21,L10,MONEY,Kshs 4,"Kshs 4, 000"


In [12]:
print("WRONG (predicted where there is no gold entity):", (~pred_p["matched"]).sum())

WRONG (predicted where there is no gold entity): 0


## 5. How far to trust these numbers

- **The sample is small.** With 37 gold entities, each one moves recall by about 3 points. A single extra spaced-ordinal date would noticeably change the DATE row. Treat these as rough estimates, not precise measurements.
- **Recall is probably too high.** In the whole cleaned slice, 60% of chunks (228 of 381) have no baseline match, but the sample has only 40% (8 of 20). Four of those 8 turned out to contain entities the baseline missed, so the chunks where the baseline finds nothing are where most misses are, and they are under-represented here.
- **Precision is probably too high.** No false positives appeared, but 25 predictions is too few to rule them out. The EDA already showed a trap (`KESC` near money patterns), and statutes written like cases (e.g. `Petition No. 6 of 2012`) would also match.
- **One labeller, no agreement check.** The labels follow a written guide, but no second person labelled the same chunks, so labelling consistency isn't measured.
- **The sample doesn't cover every court.** It is mostly Magistrates' Courts, like the slice, and it includes two pairs of chunks from the same judgment (L05/L19 and L12/L13).